In [1]:
import pandas as pd
import sys
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))
from multicor_fa.mcfa_model import fit

In [2]:
%load_ext autoreload

In [3]:
%autoreload 2

In [4]:
import torch
torch.set_default_dtype(torch.float64)

The goal of this notebook is to test an externally-simulated dataset with missing values in the MCFA pipeline.

### Load in Intersim data
Chalise, P., Raghavan, R., & Fridley, B. L. (2016). InterSIM: Simulation tool for multiple integrative 'omic datasets'. Computer methods and programs in biomedicine, 128, 69–74. [https://doi.org/10.1016/j.cmpb.2016.02.011](https://doi.org/10.1016/j.cmpb.2016.02.011)

In [5]:
cluster_df = pd.read_csv(
    '../../sim_data/clustering_assignments.tsv', sep='\t', index_col=1
)
cluster_df.head()

,subjects,cluster.id
subject1,1,2
subject2,2,1
subject3,3,5
subject4,4,3
subject5,5,2


In [6]:
exp_data = pd.read_csv('../../sim_data/expression_data.tsv', sep='\t', index_col=0)
methyl_data = pd.read_csv('../../sim_data/methylation_data.tsv', sep='\t', index_col=0)
protein_data = pd.read_csv('../../sim_data/protein_data.tsv', sep='\t', index_col=0)

# Limit to first 1000 samples for now
exp_data = exp_data.iloc[:1000,:]
methyl_data = methyl_data.iloc[:1000,:]
protein_data = protein_data.iloc[:1000,:]
cluster_df = cluster_df.iloc[:1000]

In [7]:
# Create data dictionary
Y = {
    'exp': exp_data, 
    'methyl': methyl_data,
    'prot': protein_data
}

### Fitting model to complete data

In [8]:
for name, df in Y.items():
    print(f"Missing values in {name}:", df.isna().sum().sum())

Missing values in exp: 0
Missing values in methyl: 0
Missing values in prot: 0


In [9]:
print(cluster_df.shape, Y['exp'].shape, Y['methyl'].shape, Y['prot'].shape)

(1000, 2) (1000, 131) (1000, 367) (1000, 160)


In [10]:
%%time
mcfa_res_full = fit(Y, missing_modes='raise')

Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.
There are 6 components above rho inclusion threshold 1.197622669195853.
Fitting the model.
iter: 0 Likelihood: 29821.105760421276
Iter: 1 Likelihood: 29097.54813567292 Percent change: 0.024866618361609995 Time (s): 0.00029921531677246094
Iter: 2 Likelihood: 29006.733209858747 Percent change: 0.0031308222527902177 Time (s): 0.0005199909210205078
Iter: 3 Likelihood: 28982.97358877576 Percent change: 0.000819778585182453 Time (s): 0.0007281303405761719
Iter: 4 Likelihood: 28974.306920661005 Percent change: 0.00029911563160029894 Time (s): 0.0009260177612304688
Iter: 5 Likelihood: 28970.037121435904 Percent change: 0.00014738673641332973 Time (s): 0.0011222362518310547
Iter: 6 Likelihood: 28967.188647686675 Percent change: 9.833449092604625e-05 Time (s): 0.0013172626495361328
Iter: 7 Likelihood: 28964.933582268906 Percent change: 7.785501773596148e-05 Time (s): 0.001513004302978

In [11]:
full_ww = (mcfa_res_full.W['exp'] @ mcfa_res_full.W['exp'].T).values[:5,:5].round(3)
full_phi = (mcfa_res_full.Phi['exp'].values).round(3)

### Fitting model to missing data
Each sample missing maximum 1 mode

In [12]:
Y_miss = Y.copy()
Y_miss['exp'] = Y_miss['exp'].iloc[20:]
Y_miss['methyl'] = Y_miss['methyl'].drop(
    index=Y_miss['methyl'].iloc[100:125].index.tolist()
)
Y_miss['prot'] = Y_miss['prot'].drop(
    index=Y_miss['prot'].iloc[500:530].index.tolist()
)

In [13]:
print(cluster_df.shape, Y_miss['exp'].shape, 
      Y_miss['methyl'].shape, Y_miss['prot'].shape)

(1000, 2) (980, 131) (975, 367) (970, 160)


####  Raise error for missing modes

In [14]:
%%time
# Confirm that missing_modes = raise works
mcfa_res_miss_raise = fit(Y_miss, missing_modes='raise')

ValueError: Missing modes detected for some samples.

#### Impute mean

In [15]:
mcfa_res_miss_mean = fit(Y_miss, missing_modes='impute_mean', ll_exact=True)

Missing modes detected for some samples, Imputing with the mean
Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.


/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:307: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:307: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:307: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA e

There are 6 components above rho inclusion threshold 1.1971829017114972.
Fitting the model.
iter: 0 Likelihood: 36969.18813576222
Iter: 1 Likelihood: 36427.89059810047 Percent change: 0.014859425807377735 Time (s): 0.048397064208984375
Iter: 2 Likelihood: 36386.85966041287 Percent change: 0.0011276306356341782 Time (s): 0.09704709053039551
Iter: 3 Likelihood: 36381.17224584259 Percent change: 0.00015632851332689516 Time (s): 0.14539623260498047
Iter: 4 Likelihood: 36379.41091307724 Percent change: 4.8415648333440824e-05 Time (s): 0.19177603721618652
Iter: 5 Likelihood: 36378.46537525729 Percent change: 2.59916906938081e-05 Time (s): 0.23834514617919922
Iter: 6 Likelihood: 36377.83612752614 Percent change: 1.7297558022553503e-05 Time (s): 0.28424596786499023
Iter: 7 Likelihood: 36377.3829880829 Percent change: 1.2456625683875398e-05 Time (s): 0.32999110221862793
Iter: 8 Likelihood: 36377.04510701166 Percent change: 9.288304485546151e-06 Time (s): 0.3758580684661865
Iter: 9 Likelihood: 3

In [16]:
mean_ww = (mcfa_res_miss_mean.W['exp'] @ mcfa_res_miss_mean.W['exp'].T).values[:5,:5].round(3)
mean_phi = (mcfa_res_miss_mean.Phi['exp'].values).round(3)

#### Drop samples with missing data

In [17]:
mcfa_res_miss_drop = fit(Y_miss, missing_modes='drop', ll_exact=True)

Missing modes detected for some samples, dropping samples with missing modes. There are 925 samples remaining.
Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.
There are 6 components above rho inclusion threshold 1.2081654683319352.
Fitting the model.
iter: 0 Likelihood: 27592.75104490249
Iter: 1 Likelihood: 26924.338111388253 Percent change: 0.0248256031679945 Time (s): 0.04510807991027832
Iter: 2 Likelihood: 26840.194085529092 Percent change: 0.0031350006483197284 Time (s): 0.09003901481628418
Iter: 3 Likelihood: 26818.069866885995 Percent change: 0.0008249743084760723 Time (s): 0.13472795486450195
Iter: 4 Likelihood: 26809.98697399011 Percent change: 0.00030148813215482846 Time (s): 0.17884111404418945
Iter: 5 Likelihood: 26806.01672009356 Percent change: 0.00014811055062777034 Time (s): 0.2225661277770996
Iter: 6 Likelihood: 26803.38044479801 Percent change: 9.835607493538363e-05 Time (s): 0.2656691074371338
Iter: 7 Li

In [18]:
drop_ww = (mcfa_res_miss_drop.W['exp'] @ mcfa_res_miss_drop.W['exp'].T).values[:5,:5].round(3)
drop_phi = (mcfa_res_miss_drop.Phi['exp']).values.round(3)

### Impute model with approximate EM

In [19]:
%%time
mcfa_res_miss_impute = fit(Y_miss, missing_modes='impute_model_approx', ll_exact=True)

Missing modes detected in input, they will be imputed during model fitting using approximate EM.
Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.


/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:326: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:326: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:326: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA e

There are 6 components above rho inclusion threshold 1.199679135016563.
Fitting the model.
iter: 0 Likelihood: nan
Iter: 1 Likelihood: 34914.5182168513 Percent change: nan Time (s): 0.0490260124206543
Iter: 2 Likelihood: 34371.97648891354 Percent change: 0.015784420430775964 Time (s): 0.09749794006347656
Iter: 3 Likelihood: 33805.93033924752 Percent change: 0.01674398970789042 Time (s): 0.14587998390197754
Iter: 4 Likelihood: 33443.36376266981 Percent change: 0.010841211402975347 Time (s): 0.19266915321350098
Iter: 5 Likelihood: 33241.560241842206 Percent change: 0.006070819761750792 Time (s): 0.23917889595031738
Iter: 6 Likelihood: 33123.84493376611 Percent change: 0.003553793598281834 Time (s): 0.2848019599914551
Iter: 7 Likelihood: 33045.25091756439 Percent change: 0.0023783755311096574 Time (s): 0.33091282844543457
Iter: 8 Likelihood: 32985.9298580976 Percent change: 0.0017983746319107702 Time (s): 0.37668609619140625
Iter: 9 Likelihood: 32937.72619495146 Percent change: 0.00146347

In [20]:
approx_ww = (mcfa_res_miss_impute.W['exp'] @ mcfa_res_miss_impute.W['exp'].T).values[:5,:5].round(3)
approx_phi = (mcfa_res_miss_impute.Phi['exp']).values.round(3)

### Impute model with exact EM

In [21]:
%%time
mcfa_res_miss_exact = fit(Y_miss, missing_modes='impute_model_exact', ll_exact=True)

Missing modes detected in input, they will be imputed during model fitting using exact EM.
Calculating data PCs.
Calculating empirical covariance.
Initializing model.
Inferring the shared dimensionality.


/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:317: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:317: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  Y = [pd.concat([Y_m, pd.DataFrame(
/Users/zsxie/Documents/Brown Lab Rotation/code/MCFA/src/multicor_fa/mcfa_model.py:317: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA e

There are 6 components above rho inclusion threshold 1.1955381796383993.
Fitting the model.
iter: 0 Likelihood: nan
Iter: 1 Likelihood: 32190.52833181354 Percent change: nan Time (s): 0.14580821990966797
Iter: 2 Likelihood: 31389.677481261002 Percent change: 0.025513191431502616 Time (s): 0.28534698486328125
Iter: 3 Likelihood: 30943.929630999377 Percent change: 0.014405017577828203 Time (s): 0.42197203636169434
Iter: 4 Likelihood: 30562.396449061693 Percent change: 0.01248374559153387 Time (s): 0.5590147972106934
Iter: 5 Likelihood: 30224.55617711989 Percent change: 0.011177675197677421 Time (s): 0.6967730522155762
Iter: 6 Likelihood: 29933.583074056012 Percent change: 0.009720623900720768 Time (s): 0.8334379196166992
Iter: 7 Likelihood: 29706.943776826516 Percent change: 0.007629169090301729 Time (s): 0.9707720279693604
Iter: 8 Likelihood: 29554.393535820273 Percent change: 0.005161677258623186 Time (s): 1.1064510345458984
Iter: 9 Likelihood: 29464.138608224905 Percent change: 0.0030

In [22]:
exact_ww = (mcfa_res_miss_exact.W['exp'] @ mcfa_res_miss_exact.W['exp'].T).values[:5,:5].round(3)
exact_phi = (mcfa_res_miss_exact.Phi['exp']).values.round(3)

In [23]:
print('WW^T:')
print(f'  {full_ww[:2,:2]}')
print(f'  {mean_ww[:2,:2]}')
print(f'  {drop_ww[:2,:2]}')
print(f'  {approx_ww[:2,:2]}')
print(f'  {exact_ww[:2,:2]}')

WW^T:
  [[ 0.801 -0.064]
 [-0.064  0.826]]
  [[ 0.799 -0.068]
 [-0.068  0.815]]
  [[ 0.807 -0.063]
 [-0.063  0.828]]
  [[ 0.417 -0.032]
 [-0.032  0.428]]
  [[ 0.786 -0.034]
 [-0.034  0.78 ]]


In [24]:
print('Phi:')
print(f'  {full_phi[:2,:2]}')
print(f'  {mean_phi[:2,:2]}')
print(f'  {drop_phi[:2,:2]}')
print(f'  {approx_phi[:2,:2]}')
print(f'  {exact_phi[:2,:2]}')

Phi:
  [[0.341 0.022]
 [0.022 0.36 ]]
  [[ 0.812 -0.067]
 [-0.067  0.451]]
  [[0.339 0.023]
 [0.023 0.364]]
  [[0.655 0.017]
 [0.017 0.545]]
  [[0.336 0.055]
 [0.055 0.055]]
